# Data Integration: U.S. Interest Rates and Housing Market

This notebook combines the four cleaned datasets into one monthly dataset. I checked data types and duplicate dates, merged the datasets by date, and verified the final dataset before saving it for analysis.

In [28]:
import pandas as pd
from pathlib import Path

## 1. Loading Cleaned Datasets

I loaded the four cleaned CSV files containing federal funds rates, mortgage rates, housing starts, and new home sales.

In [29]:
processed_data = Path("../data/processed")

In [30]:
fed_funds = pd.read_csv(processed_data / "fed_funds_clean.csv")
mortgage = pd.read_csv(processed_data / "mortgage_clean.csv")
housing_starts = pd.read_csv(processed_data / "housing_starts_clean.csv")
new_home_sales = pd.read_csv(processed_data / "new_home_sales_clean.csv")

In [31]:
fed_funds["date"] = pd.to_datetime(fed_funds["date"])
mortgage["date"] = pd.to_datetime(mortgage["date"])
housing_starts["date"] = pd.to_datetime(housing_starts["date"])
new_home_sales["date"] = pd.to_datetime(new_home_sales["date"])

## 2. Checking Data Types

I converted the date columns to datetime format to ensure the datasets could be merged correctly by date.

In [32]:
fed_funds.dtypes

date              datetime64[ns]
fed_funds_rate           float64
dtype: object

In [33]:
mortgage.dtypes

date             datetime64[ns]
mortgage_rate           float64
dtype: object

In [34]:
housing_starts.dtypes

date              datetime64[ns]
housing_starts             int64
dtype: object

In [35]:
new_home_sales.dtypes

date              datetime64[ns]
new_home_sales             int64
dtype: object

## 3. Checking Duplicate Dates

I checked each dataset for duplicate dates to identify potential issues before merging the data.

In [36]:
fed_funds["date"].duplicated().sum()

np.int64(0)

In [37]:
mortgage["date"].duplicated().sum()

np.int64(0)

In [38]:
housing_starts["date"].duplicated().sum()

np.int64(0)

In [39]:
new_home_sales["date"].duplicated().sum()

np.int64(0)

## 4. Merging the Datasets

I used inner joins on the date column to combine the four datasets. I then filtered the data to include observations from January 2000 onward and checked the final dataset for missing values and date coverage.

In [40]:
merge_date = (
    fed_funds
    .merge(mortgage, on="date", how="inner")
    .merge(housing_starts, on="date", how="inner")
    .merge(new_home_sales, on="date", how="inner" )
)

In [41]:
merge_date.shape

(665, 5)

In [42]:
merge_date.head()

,date,fed_funds_rate,mortgage_rate,housing_starts,new_home_sales
0,1971-04-01,4.16,7.31,1986,662
1,1971-05-01,4.63,7.43,2049,618
2,1971-06-01,4.91,7.53,2026,646
3,1971-07-01,5.31,7.60,2083,706
4,1971-08-01,5.57,7.70,2158,659


In [44]:
merge_date.tail()

,date,fed_funds_rate,mortgage_rate,housing_starts,new_home_sales
660,2026-04-01,3.64,6.33,1414,641
661,2026-05-01,3.63,6.44,1182,636
662,2026-06-01,3.63,6.49,1439,672
663,2026-07-01,3.63,6.54,1309,643
664,2026-08-01,3.63,6.67,1275,684


In [43]:
merge_date.columns

Index(['date', 'fed_funds_rate', 'mortgage_rate', 'housing_starts',
       'new_home_sales'],
      dtype='object')

In [45]:
merge_date = merge_date[
    ( merge_date["date"] >= "2000-01-01")
].copy()

In [48]:
merge_date.isna().sum()

date              0
fed_funds_rate    0
mortgage_rate     0
housing_starts    0
new_home_sales    0
dtype: int64

In [50]:
merge_date["date"].min(), merge_date["date"].max()

(Timestamp('2000-01-01 00:00:00'), Timestamp('2026-08-01 00:00:00'))

In [52]:
merge_date.shape

(320, 5)

## 5. Saving the Merged Dataset

I saved the final merged dataset as a CSV file in the processed data folder for use in exploratory data analysis, SQL analysis, and dashboard development.

In [53]:
merge_date.to_csv(
    processed_data / "merged_housing_interest_rates_2000_2026.csv",
    index=False
)